# Trabalho Final — Restauração e Preenchimento Inteligente de Imagens

## Image Inpainting com Modelos de Difusão

**Objetivo:** desenvolver uma solução aplicada para remover uma região de uma imagem e reconstruí-la automaticamente utilizando um modelo pré-treinado de difusão para *Image Inpainting*.

**Pipeline:** imagem original → máscara → imagem degradada → modelo de inpainting → imagem reconstruída → avaliação visual e quantitativa.

> **Ambiente recomendado:** Google Colab com GPU. O modelo de difusão possui custo computacional elevado para execução em CPU.

## 1. Definição do problema

Em aplicações de processamento de imagens, é comum existir a necessidade de remover objetos, preencher regiões ausentes ou restaurar partes danificadas de uma imagem.

Métodos tradicionais de inpainting podem apresentar dificuldades quando a região removida contém estruturas complexas. Este projeto utiliza um modelo de difusão pré-treinado para gerar uma reconstrução semanticamente coerente.

### Objetivos

- Implementar um pipeline de Image Inpainting.
- Criar máscaras artificialmente para simular regiões ausentes.
- Utilizar um modelo pré-treinado de difusão.
- Comparar a imagem reconstruída com a imagem original.
- Avaliar os resultados com MAE, MSE, PSNR e SSIM.
- Analisar limitações e possibilidades de melhoria.

## 2. Abordagem escolhida

Foi escolhido o **Stable Diffusion Inpainting**, acessado pela biblioteca Hugging Face Diffusers.

A escolha é justificada por:

1. Aproveitar conhecimento visual adquirido durante o pré-treinamento.
2. Evitar o custo de treinar uma rede de difusão do zero.
3. Permitir a utilização de uma máscara para definir exatamente a região a ser reconstruída.
4. Permitir condicionamento textual por meio de um prompt.

### Fluxo

```text
Imagem original + Máscara + Prompt
                ↓
      Stable Diffusion Inpainting
                ↓
       Imagem reconstruída
                ↓
      Avaliação visual e métricas
```

## 3. Instalação das dependências

As bibliotecas principais são:

- `diffusers` — pipeline de difusão.
- `transformers` — componentes de modelos.
- `accelerate` — execução eficiente.
- `torch` — processamento do modelo.
- `Pillow` — manipulação de imagens.
- `numpy` — operações numéricas.
- `scikit-image` — métricas de avaliação.
- `matplotlib` — visualização.

In [ ]:
!pip -q install diffusers transformers accelerate safetensors scikit-image

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import torch

from diffusers import StableDiffusionInpaintPipeline
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import mean_squared_error

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("AVISO: GPU não encontrada. A execução do modelo será muito mais lenta.")

## 4. Preparação da imagem

Para tornar o experimento reproduzível, o notebook permite:

- carregar uma imagem própria;
- ou utilizar uma imagem de exemplo disponível no ambiente.

Para uma avaliação quantitativa válida, a imagem original é mantida intacta. A máscara é aplicada apenas para criar a entrada degradada.

In [ ]:
# Se estiver no Google Colab, descomente as linhas abaixo para enviar uma imagem:
# from google.colab import files
# uploaded = files.upload()
# IMAGE_PATH = next(iter(uploaded))

# Caso não envie uma imagem, será utilizada uma imagem de exemplo.
import urllib.request

IMAGE_PATH = "/content/imagem_original.jpg"

if not os.path.exists(IMAGE_PATH):
    url = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
    urllib.request.urlretrieve(url, IMAGE_PATH)

image = Image.open(IMAGE_PATH).convert("RGB")
print("Tamanho original:", image.size)

display(image)

## 5. Pré-processamento

Modelos de difusão possuem custo computacional elevado. Por isso, a imagem é redimensionada para uma resolução controlada.

A resolução `512 × 512` é adequada para demonstrar o método, mas pode ser reduzida em GPUs com pouca memória.

In [ ]:
IMAGE_SIZE = (512, 512)

image = image.resize(IMAGE_SIZE, Image.Resampling.LANCZOS)
image.save("/content/imagem_processada.png")

display(image)

## 6. Criação da máscara

A máscara indica ao modelo quais pixels devem ser reconstruídos.

- **0 (preto):** preservar a imagem original.
- **255 (branco):** região a ser preenchida.

Neste experimento é criada uma máscara retangular. O tamanho e a posição podem ser alterados para testar diferentes níveis de dificuldade.

In [ ]:
def create_rectangular_mask(size, box):
    mask = Image.new("L", size, 0)
    draw = ImageDraw.Draw(mask)
    draw.rectangle(box, fill=255)
    return mask

# Região removida — altere estes valores para novos experimentos.
MASK_BOX = (180, 170, 340, 330)

mask = create_rectangular_mask(image.size, MASK_BOX)
mask.save("/content/mask.png")

# Criar visualização da região degradada
damaged = image.copy()
damaged_pixels = np.array(damaged)
mask_array = np.array(mask)

# Neutraliza a região mascarada para facilitar a visualização da entrada.
damaged_pixels[mask_array > 0] = 0
damaged = Image.fromarray(damaged_pixels)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image)
axes[0].set_title("Imagem original")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Máscara")
axes[2].imshow(damaged)
axes[2].set_title("Imagem degradada")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## 7. Carregamento do modelo

O pipeline utilizado é o `StableDiffusionInpaintPipeline`.

A execução utiliza `float16` quando uma GPU CUDA está disponível, reduzindo o consumo de memória.

> Dependendo da versão do Diffusers e da disponibilidade do modelo, o identificador utilizado abaixo pode exigir autenticação ou apresentar incompatibilidade. Nesse caso, pode-se substituir por outro checkpoint compatível com Stable Diffusion Inpainting.

In [ ]:
MODEL_ID = "runwayml/stable-diffusion-inpainting"

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    safety_checker=None
)

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = pipe.to(device)

# Otimizações quando disponíveis
if device == "cuda":
    pipe.enable_attention_slicing()

print("Modelo carregado em:", device)

## 8. Geração da imagem reconstruída

O prompt orienta o modelo sobre o conteúdo esperado na região mascarada.

Para uma avaliação mais justa, é importante manter o mesmo prompt e os mesmos parâmetros ao comparar diferentes máscaras.

In [ ]:
PROMPT = "a realistic continuation of the original image, natural lighting, coherent details"
NEGATIVE_PROMPT = "blurry, distorted, deformed, unrealistic, artifacts"

SEED = 42
generator = torch.Generator(device=device).manual_seed(SEED)

result = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    image=image,
    mask_image=mask,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator
).images[0]

result.save("/content/imagem_inpainting.png")

display(result)

## 9. Comparação visual

A comparação visual permite verificar se a região mascarada foi preenchida de maneira coerente com o restante da imagem.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(image)
axes[0].set_title("Original")

axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Máscara")

axes[2].imshow(damaged)
axes[2].set_title("Entrada degradada")

axes[3].imshow(result)
axes[3].set_title("Resultado do Inpainting")

for ax in axes:
    ax.axis("off")

plt.tight_layout()

## 10. Avaliação quantitativa

Como conhecemos a imagem original antes de aplicar a máscara, podemos comparar a região reconstruída com o conteúdo original.

Serão utilizadas:

- **MAE:** erro absoluto médio. Quanto menor, melhor.
- **MSE:** erro quadrático médio. Quanto menor, melhor.
- **PSNR:** relação sinal-ruído em decibéis. Quanto maior, melhor.
- **SSIM:** similaridade estrutural. Valores mais próximos de 1 indicam maior similaridade.

A avaliação deve ser interpretada com cuidado: em inpainting generativo, uma reconstrução pode ser visualmente plausível e semanticamente correta sem reproduzir exatamente os mesmos pixels da imagem original.

In [ ]:
original_np = np.asarray(image).astype(np.float32) / 255.0
result_np = np.asarray(result).astype(np.float32) / 255.0

mask_np = np.asarray(mask) > 0

# Avaliação apenas na região reconstruída
original_region = original_np[mask_np]
result_region = result_np[mask_np]

mae = np.mean(np.abs(original_region - result_region))
mse = np.mean((original_region - result_region) ** 2)

# SSIM/PSNR na imagem completa
psnr_value = psnr(original_np, result_np, data_range=1.0)
ssim_value = ssim(original_np, result_np, channel_axis=2, data_range=1.0)

print(f"MAE na região mascarada: {mae:.6f}")
print(f"MSE na região mascarada: {mse:.6f}")
print(f"PSNR na imagem completa: {psnr_value:.2f} dB")
print(f"SSIM na imagem completa: {ssim_value:.4f}")

## 11. Avaliação focada na região reconstruída

As métricas calculadas na região mascarada são particularmente importantes, pois a área fora da máscara permanece praticamente inalterada.

Para SSIM e PSNR focados exclusivamente na região, recortamos a caixa utilizada na máscara.

In [ ]:
x1, y1, x2, y2 = MASK_BOX

orig_crop = original_np[y1:y2, x1:x2]
res_crop = result_np[y1:y2, x1:x2]

psnr_region = psnr(orig_crop, res_crop, data_range=1.0)
ssim_region = ssim(orig_crop, res_crop, channel_axis=2, data_range=1.0)

print(f"PSNR na região reconstruída: {psnr_region:.2f} dB")
print(f"SSIM na região reconstruída: {ssim_region:.4f}")

## 12. Análise dos resultados

A interpretação das métricas deve ser combinada com a inspeção visual.

### Pontos a observar

- O conteúdo gerado combina com o contexto ao redor?
- Existem bordas artificiais?
- O modelo criou objetos que não existiam?
- Houve perda de textura?
- A iluminação e as cores permanecem coerentes?
- O resultado é semanticamente plausível mesmo quando os pixels não são idênticos ao original?

Essa análise é importante porque o objetivo do inpainting generativo não é necessariamente recuperar exatamente os pixels originais, mas produzir uma continuação visual coerente.

## 13. Experimento adicional — diferentes tamanhos de máscara

Um experimento útil é aumentar a região removida.

Quanto maior a região desconhecida, maior é a quantidade de informação que o modelo precisa inferir.

In [ ]:
# Apenas prepara três máscaras para comparação.
# Para reduzir o custo computacional, a geração pode ser executada individualmente.

mask_sizes = {
    "Pequena": (220, 210, 300, 290),
    "Média": (180, 170, 340, 330),
    "Grande": (130, 120, 390, 380),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, box) in zip(axes, mask_sizes.items()):
    m = create_rectangular_mask(image.size, box)
    ax.imshow(m, cmap="gray")
    ax.set_title(name)
    ax.axis("off")

plt.tight_layout()

## 14. Limitações

O método apresenta algumas limitações:

1. **Custo computacional:** modelos de difusão exigem bastante memória e tempo de processamento.
2. **Dependência do prompt:** instruções textuais inadequadas podem produzir resultados inconsistentes.
3. **Dependência da máscara:** máscaras maiores tornam a reconstrução mais difícil.
4. **Não garante fidelidade:** o modelo pode gerar conteúdo plausível, mas diferente do conteúdo original.
5. **Artefatos:** podem surgir distorções, texturas artificiais ou objetos inexistentes.
6. **Avaliação:** métricas pixel a pixel podem penalizar uma imagem visualmente boa quando a reconstrução legítima é diferente da original.

## 15. Possibilidades de melhoria

Como trabalhos futuros, podem ser investigadas:

- comparação entre diferentes modelos de inpainting;
- utilização de ControlNet para maior controle estrutural;
- teste de diferentes prompts;
- experimentos com diferentes tamanhos e formatos de máscara;
- comparação com métodos clássicos de OpenCV;
- avaliação perceptual utilizando métricas específicas;
- utilização de conjuntos de imagens maiores;
- fine-tuning ou LoRA para um domínio específico;
- execução em resolução maior quando houver GPU suficiente.

## 16. Conclusão

O projeto demonstra uma aplicação de IA generativa para processamento e geração de imagens utilizando Image Inpainting baseado em modelos de difusão.

A solução recebe uma imagem e uma máscara, identifica a região a ser preenchida e utiliza um modelo pré-treinado para gerar uma continuação visualmente coerente.

A avaliação combina métricas quantitativas com análise visual, permitindo identificar tanto a qualidade numérica quanto as limitações perceptuais da reconstrução.

O principal resultado é demonstrar que modelos de difusão podem ser aplicados à restauração e edição de imagens sem a necessidade de treinar um modelo completo do zero.

## 17. Resumo para apresentação

### Slide 1 — Problema, objetivo e dados
- Restauração/preenchimento de regiões ausentes.
- Imagem original + máscaras artificiais.
- Objetivo: reconstruir a região removida com IA generativa.

### Slide 2 — Solução
- Stable Diffusion Inpainting.
- Python + PyTorch + Diffusers.
- Entrada: imagem + máscara + prompt.
- Saída: imagem reconstruída.

### Slide 3 — Resultados
- Mostrar Original → Máscara → Degradada → Resultado.
- Apresentar MAE, MSE, PSNR e SSIM.
- Discutir artefatos, dependência da máscara/prompt e custo computacional.
- Concluir com possibilidades de melhoria.